In [1]:
import pandas as pd
import numpy as np
import torch

from matplotlib import pyplot
from mpl_toolkits.mplot3d import Axes3D

In [2]:
device = torch.device("cpu")

path = "dataset/HMCC.csv"
df = pd.read_csv(path)

X_raw = df.iloc[:, 1:].values
y_raw = df.iloc[:, 0].values

print(X_raw.shape)
print(y_raw.shape)

(346499, 784)
(346499,)


In [3]:
m = X_raw.shape[0]
train_size = int(0.8 * m)

In [10]:
x_2=np.array(X_raw)
y_2=np.array(y_raw)

X_train =x_2[:train_size] / 255.
X_test =x_2[train_size:] / 255.

y_train = y_2[:train_size].astype(np.int32)
y_test = y_2[train_size:].astype(np.int32)

m = X_train.shape[0] 

X_t = torch.from_numpy(X_train).float().to(device)
y_t = torch.from_numpy(y_train).long().to(device)

# 5. Verificación de dimensiones y tipos
#print(f"Total de clases: {len(torch.unique(y_t.shape))}")
print(f"X_train shape: {X_t.shape} | Dispositivo: {X_t.device}")
print(f"X_test shape:  {X_test.shape}  | Dispositivo: {X_test.device}")
print(f"Y_train shape: {y_t.shape} | Dispositivo: {y_t.device}")
print(f"Y_test shape: {y_test.shape} | Dispositivo: {y_test.device}")

X_train shape: torch.Size([277199, 784]) | Dispositivo: cpu
X_test shape:  (69300, 784)  | Dispositivo: cpu
Y_train shape: torch.Size([277199]) | Dispositivo: cpu
Y_test shape: (69300,) | Dispositivo: cpu


In [7]:
from sklearn.metrics import accuracy_score

# Función softmax manual creada por tu ingeniero
def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1, keepdims=True)

# Función de evaluación usando el modelo y la softmax manual
def evaluate(x):
    model.eval()
    with torch.no_grad():
        y_pred = model(x)
        y_probas = softmax(y_pred)
        return torch.argmax(y_probas, axis=1)

In [ ]:
# Configuración de dimensiones
D_in, H1, D_out = 784, 132, 66

# 1. Definición del modelo Multicapa
model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H1),
    torch.nn.ReLU(),
    torch.nn.Linear(H1, D_out)
).to(device)

# 2. Pérdida y Optimizador
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# 3. Bucle de Entrenamiento
epochs = 200
log_each = 20
l = []

model.train()
for e in range(1, epochs + 1):

    # Forward pass
    y_pred = model(X_t)

    # Loss
    loss = criterion(y_pred, y_t)
    l.append(loss.item())

    # Ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop
    loss.backward()

    # Update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cpu())
acc = accuracy_score(y_test, y_pred.cpu().numpy())

print(f"\nAccuracy final en Test: {acc * 100:.2f}%")

Epoch 20/200 Loss 4.17565
Epoch 40/200 Loss 4.15398
Epoch 60/200 Loss 4.12852
Epoch 80/200 Loss 4.09761
Epoch 100/200 Loss 4.06033
Epoch 120/200 Loss 4.01663
Epoch 140/200 Loss 3.96733
Epoch 160/200 Loss 3.91391
Epoch 180/200 Loss 3.85794
Epoch 200/200 Loss 3.80081

Accuracy final en Test: 27.89%
